In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import utulek

utulek.platform.import_globals_notebook(globals())

base_module = utulek
IS_NOTEBOOK_KERNEL_CODE = False
NOTEBOOK_NAME = 2026_07_27-t2t-65a0110d-2fe4-43e9-af6f-c395776e1529.ipynb
ASSET_PATH = /home/gilgamesh/main.syncthing/utulek/2026_07_27-t2t-65a0110d-2fe4-43e9-af6f-c395776e1529.ipynb.asset/
torch: ['NVIDIA GeForce GTX 1080 Ti']
tf: /device:GPU:0
jax: [CudaDevice(id=0)]


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(
	checkpoint,
	dtype="auto",
	device_map="auto",
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [4]:
tokenizer.response_schema = {
	"x-regex":
	r"^(?:(?:<think>)?\s*(?P<thinking>.+?)\s*</think>)?\s*(?:<tool_call>(?P<tool_calls>.*?)\s*</tool_call>)?\s*(?P<content>.+?)?\s*(?:<\|im_end\|>|$)",
	"type": "object",
	"properties": {
		"role": {
			"const": "assistant"
		},
		"content": {
			"type": "string"
		},
		"thinking": {
			"type": "string"
		},
		"tool_calls": {
			"x-regex-iterator": r"^(.*)$",
			"type": "array",
			"items": {
				"type": "object",
				"properties": {
					"type": {
						"const": "function"
					},
					"function": {
						"x-parser": "json",
						"x-parser-args": {
							"allow_non_json": True
						},
						"type": "object",
						"properties": {
							"name": {
								"type": "string"
							},
							"arguments": {
								"type": "object",
								"additionalProperties": {}
							},
						},
					},
				},
			},
		},
	},
}

In [5]:
from transformers import TextIteratorStreamer
from threading import Thread


def prompt(
		user_prompt: str,
		history=[
			{
				"role":
				"system",
				"content":
				"You are a general-purpose technical assistant."
			}
		],
		max_new_tokens: int = 8192,
		enable_thinking: bool = False):
	messages = history
	messages.append({"role": "user", "content": user_prompt})
	input_ids = tokenizer.apply_chat_template(
		messages,
		add_generation_prompt=True,
		return_dict=True,
		enable_thinking=enable_thinking,
		return_tensors="pt")["input_ids"].to(model.device)
	streamer = TextIteratorStreamer(
		tokenizer, skip_prompt=True, skip_special_tokens=True)
	generation_args = {
		"input_ids": input_ids,
		"max_new_tokens": max_new_tokens,
		"temperature": 0.6,
		"top_p": 0.95,
		"top_k": 20,
		"min_p": 0.0,  # "presence_penalty": 0.0,
		"repetition_penalty": 1.0,
		"do_sample": True,
		"streamer": streamer,
	}
	thread = Thread(
		target=model.generate, kwargs=generation_args)
	thread.start()
	tokens = []
	for token in streamer:
		tokens.append(token)
		yield token
	thread.join()
	history.append(
		{
			"role": "assistant", "content": "".join(tokens)
		})


history = [
	{
		"role":
		"system",
		"content":
		"You are a general-purpose technical assistant."
	}
]
torch.cuda.empty_cache()

In [13]:
model.to("cuda")

for token in prompt(
		"""
im feeling depressed
""",
		history=history,
		enable_thinking=True,
):
	print(token, end="")

with open("../snowfall/t2t.md", "w") as f:
	f.write(history[-1]["content"])

Thinking Process:

1.  **Analyze 

KeyboardInterrupt: 

In [1]:
model.to("cpu")
torch.cuda.empty_cache()

NameError: name 'model' is not defined